<div align="center" style="border:solid 1px gray;">
    <a href="https://openalex.org/">
        <img src="https://raw.githubusercontent.com/ourresearch/openalex-api-tutorials/1988d22c5499d6a1f68d85ef2902b600b555aaa5/resources/img/OpenAlex-banner.png" alt="OpenAlex banner" width="300">
    </a>
</div>

# 特定機関におけるオープンアクセス出版のモニタリング

<div style='background:#e7edf7'>
    このノートブックではOpenAlex APIをクエリして以下の質問に答えます。
    <blockquote>
        <b><i>特定の機関から最近発表された学術論文のうち、オープンアクセス（OA）であるものはいくつあるか？ そうでないものはいくつあるか？</i></b>
    </blockquote>
    この問題を徹底的に調べるために、以下のAPI機能を使用します。
    <a href="https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists">filtering</a> 及び 
    <a href="https://docs.openalex.org/how-to-use-the-api/get-groups-of-entities">grouping</a>
</div>
<br>

九州大学のオープンアクセス（OA）移行の進捗状況を追跡したい場合を想像してください。OpenAlexを使ってそれをどのように実現できますか？

### Steps
まず、そのプロセスを管理しやすい小さなステップに分けましょう。
1. まず九州大学の最近の学術論文をすべて取得します。
2. 次にそれらをオープンアクセスとクローズドアクセスに分類します。
3. 最後に各カテゴリーの出版物をカウントします。
4. さらに結果を可視化するために数値をプロットにまとめることができます。

### Input
必要なインプットは、機関を識別するための識別子だけです。ここでは、そのためにROR IDを選びました。(https://ror.org/).  
九州大学をRORレジストリで検索すると、そのROR IDは https://ror.org/00p4k0j84 であることがわかります。:

In [ ]:
#input
ror = 'https://ror.org/00p4k0j84' #九州大学

準備は万端、さあ始めましょう！

<hr>

## 1. 九州大学の最近の学術論文をすべて取得する
OpenAlexへのクエリ送信では、まず必要なデータを正確に取得するためのURLを構築します。以下の2点を指定する必要があります。
1. どのエンティティタイプ（著者、概念、機関、発表会場、著作物）のデータを取得するか？  
* --> 「学術論文」(journal articles)に関するメタデータを取得するため、エンティティタイプは著作物(works)とします。

2. 目的を満たすために著作物(works)が満たすべき条件は何か？  
* ここでは、[著作物(works)に利用可能なフィルター一覧](https://docs.openalex.org/api-entities/works/filter-works)を確認し、適切なものを選択する必要があります。  
* --> 「九州大学の最新の学術論文をすべて取得する」ために、次の条件でフィルタリングします。  
  * 過去5年間に出版されたもの（＝recent）：from_publication_date:2021-01-01  
  * 記事として指定されているもの：type:article  
  * 少なくとも1人の[著者(authorship)](https://docs.openalex.org/api-entities/works/work-object#authorships)が九州大学に所属しているもの：institutions.ror:https://ror.org/00p4k0j84  
  * [パラテキスト](https://docs.openalex.org/api-entities/works/work-object#is_paratext)ではないもの：is_paratext:false  

<br>

さて、これらの要素を組み合わせてURLを作成する必要があります。手順は次のとおりです。
* まず、OpenAlex APIのベースURLが出発点です： `https://api.openalex.org/`
* 次に、エンティティタイプを追加します: `https://api.openalex.org/works`
* すべての条件は、クエリパラメータfilterに入れ、これはURLの末尾に「?」の後に追加します: `https://api.openalex.org/works?filter=`
* filter の値を構築するために、指定した条件をカンマで区切って連結します:  
`https://api.openalex.org/works?filter=institutions.ror:https://ror.org/00p4k0j84,type:article,from_publication_date:2021-01-01,is_paratext:false`

このURLを使えば、九州大学の最近の学術論文(journal articles)をすべて取得できます！

In [ ]:
def build_institution_works_url(ror):
    # specify endpoint
    endpoint = 'works'

    # build the 'filter' parameter
    filters = (
        f'institutions.ror:{ror}',
        'is_paratext:false',
        'type:article', 
        #'topics.domain.id:4', #Health Sciences
        #'topics.field.id:27', #Medicine
        'from_publication_date:2021-01-01', 
        'to_publication_date:2025-12-31'
    )
    
    # put the URL together
    return f'https://api.openalex.org/{endpoint}?filter={",".join(filters)}'

filtered_works_url = build_institution_works_url(ror)
print(f'complete URL with filters:\n{filtered_works_url}')

<hr>

## 2. オープンアクセスとクローズドアクセスに分類する
オープンアクセスとクローズドアクセスの論文数を取得するためには、取得した著作物をさらにこれらのカテゴリーに分けるために使える追加の属性を見つける必要があります。幸いなことに、OpenAlexは著作物のメタデータ内に、ネストされた[OpenAccessオブジェクト](https://docs.openalex.org/api-entities/works/work-object#the-openaccess-object)を通じてアクセス状況に関する情報を含んでいます。このオブジェクトは次の3つの属性で構成されています。
* `is_oa` (Boolean): この著作物がオープンアクセスであれば True
* `oa_status` (String): この著作物のオープンアクセス（OA）ステータス。取り得る値は gold、green、hybrid、bronze、closed
* `oa_url` (String): この著作物に対する最適なオープンアクセス（OA）URL

**-->`is_oa` は、まさに私たちが探している条件のようです！**


#### Shortcut `group_by`
So one way to get the number of open and closed works would be to add `is_oa` as an additional filter to our query and query OpenAlex for each value in its range `{true, false}` to get its resulting count of works, e.g.
* `filter=...,is_oa:true`
* `filter=...,is_oa:false`


But wait! Isn't that exactly what `group_by` does?  
Yes, absolutely, the `group_by` parameter takes one attribute as input, divides the list of results based on the attribute's values and returns each of their counts. What a time saver!

Let's add `group_by=is_oa` as an additional query parameter to the end of our URL:

In [ ]:
group_by_param = 'group_by=is_oa'

work_groups_url = f'{filtered_works_url}&{group_by_param}'
print(f'complete URL with group_by:\n{work_groups_url}')

<hr>

## 3. Count the number of works in each group

After putting together the URL, we can query OpenAlex for the groups of publications and retrieve the following two groups:

In [ ]:
import requests, json
response = requests.get(work_groups_url).json()

work_groups = response['group_by']
print(json.dumps(work_groups, indent=2))

Each group is made up of its `key` that contains the attribute value for the `group_by` attribute, in our case `is_oa`, and its `count` of entities belonging to the group. Given these data we can already answer our initial question:  
> _How many of recent journal articles from a given institution are Open Access? And how many aren't?_

In [ ]:
def calculate_open_closed_counts(work_groups):
    open_works_count = 0
    closed_works_count = 0
    for index, group in enumerate(work_groups):
        print(f"--> Group {index+1} includes all works where `is_oa` is {group['key']} and has a count of {group['count']} publications.")

        if group['key_display_name']=="true":
            open_works_count += group['count']
        else: 
            closed_works_count += group['count']
    
    return open_works_count, closed_works_count

open_works_count, closed_works_count = calculate_open_closed_counts(work_groups)
total_works_count = open_works_count + closed_works_count

if total_works_count > 0:
    print('That makes an OA percentage of %f' % (100 * open_works_count/total_works_count))
else:
    print('OA percentage can`t be determined, no publications in result')

<hr>

## 4. Plot the data (optional)
Last but not least we can put the data into a visually appealing plot. How about a donut plot?

In [ ]:
def create_donut_plot(open_works_count, closed_works_count):
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (8,5.5)

    # set labels and their respective values
    groups = ['Open Access', 'Closed Access']
    counts = [open_works_count, closed_works_count]

    # some visual settings
    colors = ['#23c552', '#f84f31']
    explode = (0.01, 0.01)

    # pie chart
    plt.pie(counts, colors=colors, labels=groups,
            autopct='%1.1f%%', pctdistance=0.85,
            explode=explode, textprops={'fontsize': 14})

    # make it a donut (draw circle in the middle)
    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)
    
    # display chart
    plt.show()

# create donut chart from open/closed counts
create_donut_plot(open_works_count, closed_works_count)

---
Feel free to use the notebook and determine the percentage of Open Access works for your institution or tweak the filters to fit your analysis.  

Happy exploring! 😎